# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/interim/train.csv")
test = pd.read_csv("../data/interim/test.csv")

train.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,NEAR BAY
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,<1H OCEAN
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,INLAND
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,INLAND
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,NEAR OCEAN


**MISSING VALUES**

La variable `total_bedrooms` presenta valores faltantes (168 observaciones), lo que constituye un problema de completitud que podría afectar la calidad del modelo.
Dado que esta variable está directamente relacionada con la estructura de la vivienda (`total_rooms`), se opta por una imputación mediante la mediana. Esta elección se justifica porque la mediana es robusta frente a valores extremos y permite preservar la distribución original de los datos sin introducir sesgos artificiales.
Esta decisión evita la pérdida de información y mantiene la coherencia estructural del dataset, asegurando que las relaciones entre variables no se vean distorsionadas.

In [2]:
train.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        168
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [11]:
median_bedrooms = train["total_bedrooms"].median()

train["total_bedrooms"] = train["total_bedrooms"].fillna(median_bedrooms)
test["total_bedrooms"] = test["total_bedrooms"].fillna(median_bedrooms)

**FEATURE ENGINEERING**

Se construyen variables derivadas con el objetivo de transformar magnitudes absolutas en métricas relativas que reflejen mejor las condiciones reales de los distritos.
Las variables originales como `total_rooms` o `population` pueden ser difíciles de interpretar de forma aislada, ya que no consideran la escala del hogar o la densidad poblacional. Por ello, se generan nuevas variables que capturan relaciones estructurales clave:

- `rooms_per_household`: mide el espacio promedio por hogar.
- `population_per_household`: actúa como un proxy de densidad o hacinamiento.
- `bedrooms_per_room`: refleja la composición interna de la vivienda.

Adicionalmente, se incorporan nuevas métricas que enriquecen el modelo:

- `bedrooms_per_household`: aproxima la capacidad habitacional por hogar.
- `rooms_per_person`: mide el espacio disponible por individuo.
- `income_x_rooms_per_household`: captura una interacción entre ingreso y espacio, permitiendo modelar relaciones no lineales.

Estas transformaciones permiten representar de manera más precisa la estructura socioeconómica y habitacional de los distritos. 

A partir del análisis descriptivo de las variables generadas, se observa que las métricas derivadas presentan distribuciones más estables y comparables que las variables originales, reduciendo el efecto de escalas heterogéneas entre distritos. Esto sugiere que dichas variables capturan de manera más eficiente patrones estructurales relevantes.
En particular, variables como `rooms_per_household` y `population_per_household` muestran concentraciones en rangos interpretables, mientras que la variable de interacción `income_x_rooms_per_household` introduce una dimensión no lineal que podría mejorar la capacidad predictiva del modelo.
En línea con la literatura, este tipo de transformaciones suele tener un impacto más significativo en el desempeño del modelo que la elección del algoritmo, al facilitar la identificación de relaciones subyacentes en los datos.

In [28]:
def add_features(df):
    df = df.copy()
    
    # Features base
    df["rooms_per_household"] = df["total_rooms"] / df["households"]
    df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
    df["population_per_household"] = df["population"] / df["households"]
    
    # Nuevas features óptimas
    df["bedrooms_per_household"] = df["total_bedrooms"] / df["households"]
    df["rooms_per_person"] = df["total_rooms"] / df["population"]
    df["income_x_rooms_per_household"] = df["median_income"] * df["rooms_per_household"]
    
    return df

train = add_features(train)
test = add_features(test)

In [30]:
train[["rooms_per_household", "population_per_household", "bedrooms_per_room", "bedrooms_per_household", "rooms_per_person", "income_x_rooms_per_household"]].describe()

,rooms_per_household,population_per_household,bedrooms_per_room,bedrooms_per_household,rooms_per_person,income_x_rooms_per_household
count,16512.000000,16512.000000,16512.000000,16512.000000,16512.000000,16512.000000
mean,5.441010,2.995974,0.213727,1.103538,1.980071,22.589699
std,2.574143,4.457373,0.066077,0.549831,1.187303,18.090334
min,0.888889,0.692308,0.037066,0.120925,0.018065,1.131556
25%,4.443636,2.433426,0.175058,1.005988,1.522060,11.669260
50%,5.235573,2.822316,0.203120,1.048892,1.936927,18.036986
75%,6.053843,3.286385,0.239844,1.100235,2.295116,27.929290
max,141.909091,502.461538,2.818182,34.066667,55.222222,612.966667


El análisis estadístico de las variables derivadas evidencia una reducción en la dispersión relativa en comparación con las variables originales, así como una mejor concentración alrededor de la mediana.
Por ejemplo, `rooms_per_household` presenta una distribución más controlada en comparación con `total_rooms`, lo que permite una interpretación más consistente entre distritos. De manera similar, `population_per_household` reduce la variabilidad extrema observada en `population`, capturando de forma más precisa la densidad habitacional.
Esto confirma que las variables generadas no solo transforman los datos, sino que aportan información estructural relevante, lo cual es clave para mejorar el desempeño del modelo en la siguiente fase.

In [31]:
print(train.columns)

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'rooms_per_household', 'bedrooms_per_room',
       'population_per_household', 'ocean_proximity_INLAND',
       'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY',
       'ocean_proximity_NEAR OCEAN', 'bedrooms_per_household',
       'rooms_per_person', 'income_x_rooms_per_household'],
      dtype='object')


In [32]:
train.shape

(16512, 19)

Tras el proceso de feature engineering y codificación, el dataset incrementa su dimensionalidad (de 10 a 19 variables), reflejando una mayor riqueza de información disponible.
Este incremento permite capturar relaciones más complejas entre variables, especialmente a través de métricas relativas e interacciones. Sin embargo, se mantiene un enfoque controlado en la generación de nuevas features, priorizando interpretabilidad y relevancia económica, con el objetivo de evitar redundancia y reducir el riesgo de sobreajuste.
La validación del impacto de estas variables se realizará en la fase de modelado mediante métricas fuera de muestra.

**ENCODING CATEGORICO**

La variable ocean_proximity se transforma mediante one-hot encoding, dado que se trata de una variable nominal sin orden inherente. Este enfoque evita imponer relaciones artificiales entre categorías, lo cual ocurriría si se utilizara encoding ordinal.
Se elimina una categoría base (drop_first=True) para prevenir multicolinealidad en modelos lineales, asegurando estabilidad en la estimación de coeficientes.
Adicionalmente, se alinean los datasets de entrenamiento y prueba para garantizar consistencia en las dimensiones, evitando errores en la fase de inferencia.

In [33]:
if "ocean_proximity" in train.columns:
    train = pd.get_dummies(train, columns=["ocean_proximity"], drop_first=True)
    test = pd.get_dummies(test, columns=["ocean_proximity"], drop_first=True)

train, test = train.align(test, join="left", axis=1, fill_value=0)

In [34]:
X_train = train.drop("median_house_value", axis=1)
y_train = train["median_house_value"]

X_test = test.drop("median_house_value", axis=1)
y_test = test["median_house_value"]

**LOG TRANSFORM**

Dado que múltiples variables presentan asimetría positiva (identificada en el EDA), se aplica una transformación logarítmica para reducir el skewness y estabilizar la varianza.
En particular, la transformación sobre `median_income` permite suavizar la influencia de valores extremos, lo cual es consistente con la distribución observada en los datos. Esto facilita que el modelo capture relaciones más cercanas a la linealidad.
Este tipo de transformación es especialmente útil cuando las relaciones entre variables son multiplicativas, permitiendo que modelos lineales aproximen comportamientos no lineales de manera más eficiente.

**ESCALADO**

Se aplica estandarización (StandardScaler) para normalizar las variables numéricas, transformándolas a una distribución con media cero y varianza unitaria. Este paso es crítico en modelos basados en gradientes (como SGD) y en modelos lineales, donde la escala de las variables influye directamente en la convergencia y en la magnitud de los coeficientes.
Sin este proceso, variables con mayor escala dominarían el entrenamiento, generando un sesgo en la optimización del modelo.
Además, contribuye a mejorar la estabilidad numérica del modelo y acelera el proceso de convergencia durante el entrenamiento.

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# LOG
X_train["median_income_log"] = np.log1p(X_train["median_income"])
X_test["median_income_log"] = np.log1p(X_test["median_income"])

# SCALER
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
X_train.shape, X_test.shape

((16512, 19), (4128, 19))

El proceso de limpieza y feature engineering permitió transformar el dataset en una representación más informativa, consistente y adecuada para modelado predictivo.
Las variables derivadas incorporan relaciones estructurales clave entre espacio, densidad e ingreso, reduciendo la variabilidad no explicativa y mejorando la interpretabilidad del dataset. El análisis estadístico confirma que estas transformaciones generan distribuciones más estables y comparables, lo que facilita la identificación de patrones relevantes.
Asimismo, se mantuvo un enfoque controlado en la generación de nuevas features, priorizando aquellas con justificación económica clara, con el fin de evitar redundancia y minimizar el riesgo de sobreajuste.
En conjunto, estas decisiones establecen una base sólida para la fase de modelado, donde se evaluará empíricamente el impacto de las variables mediante validación fuera de muestra.